# bench_metrics

> Per-run metrics for the Phase 8 benchmark: recovery to the ideal free distribution, belief diversity, discoveries, free riding, find, overlap and follow times, and one flat CSV row per run.

Spec: `specs/multiagent/phase8/` (design-low-level section 2).

In [ ]:
#| default_exp bench_metrics

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.team_refs import idealFree, ifdDistanceTies, ledgerArrays, occupancy

NAN = float('nan')


def periods(world, T):
    "Full move periods (k, start, end): from a move to the next one, both within the run."
    Tm = world.Tmove
    if not Tm:
        return []
    return [(k, k * Tm, (k + 1) * Tm) for k in range(1, int(T // Tm) + 1) if (k + 1) * Tm <= T + 1e-9]


def recovery(world, L, N, T, delta, w=10):
    "Restricted mean recovery time to tie-invariant IFD distance `delta` (censored at `Tmove`), and the share reached."
    per = periods(world, T)
    if not per:
        return NAN, NAN
    times, counts = occupancy(L, N, world.nPatches, T)
    d = ifdDistanceTies(counts, idealFree(world, N)[0], world.intake, N)
    rec = []
    for _, a, b in per:
        r = np.convolve(d[(times >= a) & (times < b)], np.ones(w) / w, mode='valid')   # r[j]: mean of j .. j+w-1
        hit = np.flatnonzero(r <= delta + 1e-12)
        rec.append(float(hit[0] + w - 1) if len(hit) else float(world.Tmove))
    rec = np.array(rec)
    return float(rec.mean()), float((rec < world.Tmove).mean())


def trueRates(world):
    "Solo reward rate per location at the run's end: min(c, intake) at a patch, -dangerCost at danger, else 0."
    v = np.zeros(world.L)
    v[sorted(world.danger)] = -world.dangerCost
    for p, j in enumerate(world.site):
        v[int(j)] = min(world.c, world.intake[p])
    return v


def beliefs(world, estimates):
    "Belief diversity and belief error from per-agent {'mean', 'n'} estimates; nan when undefined."
    if not estimates or 'mean' not in estimates[0]:
        return NAN, NAN
    M, n = np.array([e['mean'] for e in estimates], float), np.array([e['n'] for e in estimates], float)
    known = n > 0
    Mk, cnt = np.where(known, M, np.nan), known.sum(0)
    two, one = cnt >= 2, cnt >= 1
    div = float(np.nanstd(Mk[:, two], axis=0).mean()) if two.any() else NAN
    err = float(np.abs(np.nanmean(Mk[:, one], axis=0) - trueRates(world)[one]).mean()) if one.any() else NAN
    return div, err


def goalSpread(estimates):
    "Circular standard deviation (radians) of circuit agents' goal headings in world coordinates."
    phi = np.array([e['thetaG'] - 2 * np.pi * (e['c'] - e['a']) / e['nCirc'] for e in estimates])
    R = abs(np.exp(1j * phi).mean())
    return float(np.sqrt(-2 * np.log(max(R, 1e-12))))


def _harvests(L):
    "Row indices of positive-harvest stays (or circuit fixations) at patches, in ledger order."
    return np.flatnonzero((L['kind'] != 'travel') & (L['patch'] >= 0) & (L['harvest'] > 0))


def discoveryRows(world, L):
    "Rows that are discoveries: the first positive harvest at a patch site in each move period (the `Credit` rule)."
    seen, out = set(), []
    for r in _harvests(L):
        key = (int(L['location'][r]), int(L['s0'][r] // world.Tmove) if world.Tmove else 0)
        if key not in seen:
            seen.add(key)
            out.append(r)
    return np.array(out, int)


def discoveries(world, L):
    "Discoveries per agent."
    return np.bincount(L['agent'][discoveryRows(world, L)], minlength=world.nAgents)


def freeRiding(L, disc, N):
    "Correlation across agents of harvest share and discovery share (nan below 4 agents or without variance)."
    h = np.bincount(L['agent'], weights=L['harvest'], minlength=N)
    if N < 4 or h.sum() <= 0 or disc.sum() <= 0:
        return NAN
    hs, ds = h / h.sum(), disc / disc.sum()
    return float(np.corrcoef(hs, ds)[0, 1]) if hs.std() > 0 and ds.std() > 0 else NAN


def findTime(world, L, T):
    "Mean time from a patch move until the first harvest at each moved patch (censored at `Tmove`)."
    per, idx = periods(world, T), _harvests(L)
    if not per:
        return NAN
    ts, ps = L['s0'][idx], L['patch'][idx]
    out = []
    for _, a, b in per:
        for p in range(world.nPatches):
            m = (ps == p) & (ts >= a) & (ts < b)
            out.append(float(ts[m].min() - a) if m.any() else float(world.Tmove))
    return float(np.mean(out))


def overlap(world, L, N, T):
    "Mean pairwise Jaccard overlap of the locations agents stayed at, per full move period."
    per = periods(world, T)
    if N < 2 or not per:
        return NAN
    st, vals = L['kind'] != 'travel', []
    for _, a, b in per:
        m = st & (L['s0'] >= a) & (L['s0'] < b)
        sets = [set(L['location'][m & (L['agent'] == i)].tolist()) for i in range(N)]
        for i in range(N):
            for j in range(i + 1, N):
                u = len(sets[i] | sets[j])
                if u:
                    vals.append(len(sets[i] & sets[j]) / u)
    return float(np.mean(vals)) if vals else NAN


def followTime(world, L, T):
    "Mean time from a discovery until another agent harvests at that site in the same period (censored)."
    rows, idx = discoveryRows(world, L), _harvests(L)
    if not len(rows) or not world.Tmove:
        return NAN
    loc, ag, ts = L['location'][idx], L['agent'][idx], L['s0'][idx]
    out = []
    for r in rows:
        t0 = L['s0'][r]
        end = min((t0 // world.Tmove + 1) * world.Tmove, T)
        m = (loc == L['location'][r]) & (ag != L['agent'][r]) & (ts >= t0) & (ts < end)
        out.append(float(ts[m].min() - t0) if m.any() else float(end - t0))
    return float(np.mean(out))


SCALARS = ('welfare', 'perAgentSecond', 'harvest', 'dangerCost', 'firstQuarterRate', 'gini', 'coverage',
           'dangerShare', 'ifdMean', 'soYield', 'welfareLoss')


def _plain(v):
    return '' if v is None else v if isinstance(v, (int, float, str)) else str(v)


def benchMetrics(cell, world, hist, m, delta):
    "One flat row of scalars for a grid cell (`m` = the `harnessMetrics` dict)."
    cfg, N, T = m['config'], m['N'], m['T']
    L = ledgerArrays(world.ledger)
    row = dict(study=cell['study'], factor=cell['factor'], level=str(cell['level']), seed=cell['seed'], N=N, Trun=T)
    for k, v in cfg.items():
        if k == 'world':
            row.update({f'world.{a}': _plain(b) for a, b in v.items()})
        else:
            row[k] = _plain(v)
    row.update({k: float(m.get(k, NAN)) for k in SCALARS})
    c = m['comm']
    row.update(written=c['written'], sent=c['sent'], received=c['received'], events=len(hist),
               commPerWelfare=(c['written'] + c['sent']) / m['welfare'] if m['welfare'] > 0 else NAN)
    row['rmrt'], row['reached'] = recovery(world, L, N, T, delta) if cfg['agent'] != 'circuit' else (NAN, NAN)
    est = m.get('estimates')
    if cfg['agent'] == 'circuit' and est:
        row['diversity'], row['beliefError'] = goalSpread(est), NAN
    else:
        row['diversity'], row['beliefError'] = beliefs(world, est)
    disc = discoveries(world, L)
    row.update(discoveries=int(disc.sum()), freeRiding=freeRiding(L, disc, N), findTime=findTime(world, L, T),
               overlap=overlap(world, L, N, T), followTime=followTime(world, L, T))
    return row

## Tests

In [ ]:
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.bench_metrics import (periods, recovery, trueRates, beliefs, goalSpread, discoveryRows, discoveries,
                                  freeRiding, findTime, overlap, followTime, benchMetrics)
from explRL.credit import runCredit
from explRL.harness_runs import HarnessConfig, benchWorld, runHarness
from explRL.team_refs import ledgerArrays

def row(i, t, loc, patch, h, kind='stay'):
    return (i, float(t), 1.0, kind, loc, patch, float(h), float(h), float(t), float(t) + 1.0)

# recovery: one agent; on the patch for t = 45..79 (period 1 recovers at 9 s: the first 10 s window with
# at least 5 of 10 points at distance 0), off it in period 2 (censored at 40 s)
w = benchWorld(nPatches=1, nDanger=0, intake=(5.0,), patchSites=[0], dangerSites=[], Tmove=40.0)
w.reset(0, 1)
w.ledger = [row(0, t, 0, 0, 0.85) for t in range(45, 80)] + [row(0, t, 0, None, 0.0) for t in range(80, 120)]
L = ledgerArrays(w.ledger)
test_eq(periods(w, 120.0), [(1, 40.0, 80.0), (2, 80.0, 120.0)])
test_close(recovery(w, L, 1, 120.0, 0.5), (24.5, 0.5))

# discoveries, follow time, find time: agent 0 finds the patch at site 5 (t = 41), agent 1 follows 1 s later;
# a zero harvest and a non-patch stay are not discoveries; in period 2 agent 1 finds it at 81, nobody follows
w = benchWorld(nPatches=1, nDanger=0, intake=(5.0,), patchSites=[5], dangerSites=[], Tmove=40.0)
w.reset(0, 3)
w.ledger = [row(0, 41, 5, 0, 0.5), row(1, 42, 5, 0, 0.5), row(2, 43, 5, 0, 0.0), row(2, 44, 7, None, 0.0),
            row(1, 81, 5, 0, 0.5)]
L = ledgerArrays(w.ledger)
test_eq(discoveryRows(w, L).tolist(), [0, 4]); test_eq(discoveries(w, L).tolist(), [1, 1, 0])
test_close(followTime(w, L, 120.0), (1.0 + (120.0 - 81.0)) / 2)
test_close(findTime(w, L, 120.0), (1.0 + 1.0) / 2)                  # moved patch first harvested 1 s after each move
# overlap: period 1, agents 0 and 1 share site 5 and agent 2 visited {5, 7}: pairs 1, 1/2, 1/2; period 2, only
# agent 1 stayed anywhere: its two pairs score 0 (the idle pair has an empty union and is skipped)
test_close(overlap(w, L, 3, 120.0), np.mean([1.0, 0.5, 0.5, 0.0, 0.0]))

# free riding: harvest shares (1, 2, 3, 4) against discovery shares (4, 3, 2, 1) give -1; nan below 4 agents
Lf = ledgerArrays([row(i, 0, 0, 0, i + 1) for i in range(4)])
test_close(freeRiding(Lf, np.array([4, 3, 2, 1]), 4), -1.0)
assert np.isnan(freeRiding(Lf, np.array([1, 0, 0]), 3))

# beliefs: three agents know site 0 (true solo rate min(c, intake) = 0.85) with 1, 2, 3; one agent thinks
# empty site 1 pays 5; diversity uses site 0 only, error both sites
w = benchWorld(nPatches=1, nDanger=0, intake=(2.0,), patchSites=[0], dangerSites=[]); w.reset(0, 3)
est = [dict(mean=[1.0, 5.0] + [0.0] * 62, n=[1, 1] + [0] * 62), dict(mean=[2.0] + [0.0] * 63, n=[1] + [0] * 63),
       dict(mean=[3.0] + [0.0] * 63, n=[1] + [0] * 63)]
test_close(trueRates(w)[:2], [0.85, 0.0])
test_close(beliefs(w, est), (np.sqrt(2 / 3), (abs(2.0 - 0.85) + 5.0) / 2))
test_eq([np.isnan(x) for x in beliefs(w, None)], [True, True])
# goal spread: same world-frame goal although compass readings differ -> 0; opposite goals -> large
g = 2 * np.pi / 32
test_close(goalSpread([dict(thetaG=3 * g, a=0, c=0, nCirc=32), dict(thetaG=5 * g, a=1, c=3, nCirc=32)]), 0.0, eps=1e-5)
assert goalSpread([dict(thetaG=0.0, a=0, c=0, nCirc=32), dict(thetaG=np.pi, a=0, c=0, nCirc=32)]) > 3

# the ledger discovery rule agrees with the Credit object on a real run
wr, hr, cr, mr = runCredit(HarnessConfig(channel='blackboard', T=200.0), 8, 3, beta=1.0)
test_eq(discoveries(wr, ledgerArrays(wr.ledger)).tolist(), [cr.stats[i]['discovered'] for i in range(8)])

# benchMetrics: a flat row of scalars for generic and circuit teams
cell = dict(study='main', factor='main', level='-', seed=0)
for cfg, N in ((HarnessConfig(channel='blackboard', T=120.0), 8), (HarnessConfig(channel='hierarchical', T=120.0), 4),
               (HarnessConfig(agent='circuit', channel='messages', T=60.0), 3)):
    wb, hb, cb, mb = runHarness(cfg, N, 0)
    r = benchMetrics(cell, wb, hb, mb, 0.5)
    assert all(isinstance(v, (int, float, str)) for v in r.values()), r
    assert {'rmrt', 'diversity', 'freeRiding', 'commPerWelfare', 'channel', 'Trun'} <= set(r)
test_eq(np.isnan(r['rmrt']), True); assert r['diversity'] == r['diversity']    # circuit: no rmrt, goal spread

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()